# Variogram diagnostic — Stage B B1 ST kriging

Original diagnosis: the persisted sum-metric variogram in
`models/st_variogram.json` was degenerate — `spatial.var ≈ 1e-6` (pinned to the
floor), `joint.var ≈ 7.54` (~50× expected AOD variance), `k ≈ 0.035 km/h`.

**Root cause** (uncovered in §4 below): random pair sampling from the (cell ×
slot × day) cube is dominated by combinatorially common combinations — pairs
that are simultaneously far in space *and* far in time.  Short-lag bins were
essentially empty, so the optimiser had no structure to fit a spatial or
temporal sub-component to.  Sum-metric was over-specified.

**Resolution applied to `kriging.py` / `config.py`:**

1. **Structured pair sampling** — three concatenated samplers (same-cell,
   same-slot, mixed) populate the marginals densely.
2. **Krige the (AOD − CAMS) residual**, not raw AOD — CAMS absorbs the
   large-scale spatial trend, giving a more stationary field.
3. **Drop sparse bins** — bins with fewer than `B1_VARIOGRAM_MIN_BIN_PAIRS`
   (default 30) are NaN and don't enter the fit.
4. **Cap pairs at `h_t ≤ B1_VARIOGRAM_T_BAND_H`** (10 h) — Stage A has a
   structural day-night hole that warps the temporal axis.
5. **Drop sum-metric → single-component metric variogram**:
   `γ(h_s, h_t) = γ_metric( √(h_s² + (k·h_t)²) )`.

Sections 2–3 still show the *raw-pair* empirical surface that motivated this
work.  Section 4 shows the strict marginals.  Section 5 fits and inspects the
new metric variogram on residuals.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path('.').resolve()))

import kriging
from kriging import (
    collect_training_pairs,
    _empirical_st_variogram,
    fit_variogram,
    save_variogram,
    load_variogram,
    haversine_km,
    MetricVariogram,
)
from config import (
    TRAIN_START, TRAIN_END,
    B1_W_SPACE_KM, B1_W_TIME_H,
    B1_VARIOGRAM_SPEC, B1_VARIOGRAM_INIT,
    B1_VARIOGRAM_SUBSAMPLE,
    B1_VARIOGRAM_TARGET, B1_VARIOGRAM_MIN_BIN_PAIRS, B1_VARIOGRAM_T_BAND_H,
    MODELS_DIR,
)

print(f'TRAIN window: {TRAIN_START} → {TRAIN_END}')
print(f'B1_W_SPACE_KM = {B1_W_SPACE_KM}, B1_W_TIME_H = {B1_W_TIME_H}')
print(f'target = {B1_VARIOGRAM_TARGET}  '
      f'(min_bin_pairs={B1_VARIOGRAM_MIN_BIN_PAIRS}, '
      f't_band_h={B1_VARIOGRAM_T_BAND_H})')

## 1. Load training pairs from the production window

Uses the exact same `collect_training_pairs` the production fit uses — by
default this now returns the **(AOD − CAMS) residual** (controlled by
`config.B1_VARIOGRAM_TARGET`).  Pass `target_kind='aod'` to get raw AOD for
back-comparison.  Cell may take a few minutes on a long training window.

In [ ]:
# Residual (production target) and raw AOD (for back-comparison).
lat, lon, t_h, val_resid = collect_training_pairs(
    start=TRAIN_START, end=TRAIN_END,
    target_kind='aod_minus_cams',
    max_pairs=B1_VARIOGRAM_SUBSAMPLE,
    progress=tqdm,
)
print(f'Residual pool:  lat={lat.shape}, val={val_resid.shape}')
print(f'residual: min={val_resid.min():.4f}  median={np.median(val_resid):.4f}  '
      f'p99={np.quantile(val_resid, 0.99):.4f}  max={val_resid.max():.4f}  '
      f'mean={val_resid.mean():.4f}  var={val_resid.var():.4f}')

lat_raw, lon_raw, t_h_raw, val = collect_training_pairs(
    start=TRAIN_START, end=TRAIN_END,
    target_kind='aod',
    max_pairs=B1_VARIOGRAM_SUBSAMPLE,
    progress=tqdm,
)
print(f'\nRaw-AOD pool:   lat={lat_raw.shape}, val={val.shape}')
print(f'raw AOD:  min={val.min():.4f}  median={np.median(val):.4f}  '
      f'p99={np.quantile(val, 0.99):.4f}  max={val.max():.4f}  '
      f'mean={val.mean():.4f}  var={val.var():.4f}')

## 2. Empirical 2-D variogram — raw AOD

If the spatial axis (h_t≈0 row) is flat or noisy, the solver has no structure to fit a spatial component against — which is what produced the degenerate `spatial.var ≈ 1e-6`.

In [ ]:
# Empirical surface from the **raw-AOD** pool (legacy random pair sampling,
# kept for comparison with the residual surface below).
s_c, t_c, g_raw = _empirical_st_variogram(
    lat_raw, lon_raw, t_h_raw, val,
    n_mixed_pairs=B1_VARIOGRAM_SUBSAMPLE,
    min_bin_pairs=1,           # don't drop sparse bins here — we *want* to see them
)
print(f's_centres (km): {s_c.round(1)}')
print(f't_centres (h):  {t_c.round(2)}')
print(f'g_raw  populated bins: {np.isfinite(g_raw).sum()} / {g_raw.size}')
print(f'g_raw  min={np.nanmin(g_raw):.4f}  max={np.nanmax(g_raw):.4f}  '
      f'median={np.nanmedian(g_raw):.4f}')

In [ ]:
def plot_empirical(g_grid, s_centres, t_centres, title, vmax=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
    im = ax.pcolormesh(
        t_centres, s_centres, g_grid,
        shading='auto', cmap='viridis', vmax=vmax,
    )
    ax.set_xlabel('h_t (hours)')
    ax.set_ylabel('h_s (km)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='γ(h_s, h_t)')
    return im

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_empirical(g_raw, s_c, t_c, 'raw AOD — full range', ax=axes[0])
vmax_99 = np.nanquantile(g_raw, 0.99)
plot_empirical(g_raw, s_c, t_c, f'raw AOD — clipped @ p99 ({vmax_99:.3f})', vmax=vmax_99, ax=axes[1])
plt.tight_layout(); plt.show()

# Spatial-only and temporal-only slices (h_t≈0 / h_s≈0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(s_c, g_raw[:, 0], marker='o')
axes[0].set_xlabel('h_s (km)'); axes[0].set_ylabel('γ at smallest h_t bin')
axes[0].set_title('Spatial slice (h_t bin 0)')
axes[0].grid(alpha=0.3)
axes[1].plot(t_c, g_raw[0, :], marker='o', color='tab:orange')
axes[1].set_xlabel('h_t (hours)'); axes[1].set_ylabel('γ at smallest h_s bin')
axes[1].set_title('Temporal slice (h_s bin 0)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Empirical 2-D variogram — (AOD − CAMS) residual

Same structured samplers, but with CAMS subtracted at each obs point.  Compared
to §2 the variance is smaller and the surface should be flatter / shorter-range
because CAMS has already absorbed the large-scale trend that was making
everything look correlated.

In [ ]:
# Residual empirical surface — same structured samplers, target = AOD − CAMS.
s_c_res, t_c_res, g_res = _empirical_st_variogram(
    lat, lon, t_h, val_resid,
    n_mixed_pairs=B1_VARIOGRAM_SUBSAMPLE,
    min_bin_pairs=1,           # show all bins for diagnostic comparison
)
print(f'g_res populated bins: {np.isfinite(g_res).sum()} / {g_res.size}')
print(f'g_res min={np.nanmin(g_res):.4f}  max={np.nanmax(g_res):.4f}  '
      f'median={np.nanmedian(g_res):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_empirical(g_res, s_c_res, t_c_res, 'residual — full range', ax=axes[0])
vmax_99_res = np.nanquantile(g_res, 0.99)
plot_empirical(g_res, s_c_res, t_c_res, f'residual — clipped @ p99 ({vmax_99_res:.3f})',
               vmax=vmax_99_res, ax=axes[1])
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(s_c_res, g_res[:, 0], marker='o')
axes[0].set_xlabel('h_s (km)'); axes[0].set_ylabel('γ at smallest h_t bin')
axes[0].set_title('Residual: spatial slice')
axes[0].grid(alpha=0.3)
axes[1].plot(t_c_res, g_res[0, :], marker='o', color='tab:orange')
axes[1].set_xlabel('h_t (hours)'); axes[1].set_ylabel('γ at smallest h_s bin')
axes[1].set_title('Residual: temporal slice')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Strict marginal variograms + 12×12 joint heatmap

Sections 2–3 used the smallest-bin row of random pairs as proxies for the
pure-spatial and pure-temporal slices.  Those bins are centred at h_t≈0.82 h /
h_s≈6.8 km — close to zero but not zero, so they still mix in some joint
signal.

Here we compute the **strict marginal** variograms instead:

1. **γ(h_s) at h_t = 0**: only pairs that share an *exact* timestamp.
2. **γ(h_t) at h_s = 0**: only pairs that share an *exact* (lat, lon) cell.
3. **γ(h_s, h_t)** on a **12×12** bin grid (matches the production fitter).

Run for both raw AOD and (AOD − CAMS) residual to confirm that residual
kriging gives a flatter, shorter-range surface.

In [ ]:
# ── Strict marginal variograms ──────────────────────────────────────────────
# Group samples first, then form pairs only inside each group.
#   • Same-slot pairs → spatial variogram at h_t = 0
#   • Same-cell pairs → temporal variogram at h_s = 0

def _group_boundaries(keys_sorted: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    breaks = np.flatnonzero(np.diff(keys_sorted)) + 1
    return np.r_[0, breaks], np.r_[breaks, keys_sorted.size]


def same_slot_variogram(lat, lon, t_h, val, *, s_edges,
                        max_pairs_per_slot=2000, rng_seed=0):
    """γ(h_s) using only pairs that share an exact timestamp."""
    rng = np.random.default_rng(rng_seed)
    s_centres = 0.5 * (s_edges[:-1] + s_edges[1:])
    g_sum   = np.zeros(s_centres.size, dtype=np.float64)
    g_count = np.zeros(s_centres.size, dtype=np.int64)

    order = np.argsort(t_h, kind='stable')
    starts, ends = _group_boundaries(t_h[order])
    for s, e in zip(starts, ends):
        m = e - s
        if m < 2:
            continue
        idx = order[s:e]
        n_pairs = min(max_pairs_per_slot, m * (m - 1) // 2)
        ii = rng.integers(0, m, size=n_pairs)
        jj = rng.integers(0, m, size=n_pairs)
        keep = ii != jj
        a, b = idx[ii[keep]], idx[jj[keep]]
        h_s   = haversine_km(lat[a], lon[a], lat[b], lon[b])
        gamma = 0.5 * (val[a] - val[b]) ** 2
        bin_idx = np.clip(np.digitize(h_s, s_edges) - 1, 0, s_centres.size - 1)
        np.add.at(g_sum,   bin_idx, gamma)
        np.add.at(g_count, bin_idx, 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        return s_centres, np.where(g_count > 0, g_sum / g_count, np.nan), g_count


def same_cell_variogram(lat, lon, t_h, val, *, t_edges,
                        max_pairs_per_cell=2000, rng_seed=0):
    """γ(h_t) using only pairs that share an exact (lat, lon) cell."""
    rng = np.random.default_rng(rng_seed)
    t_centres = 0.5 * (t_edges[:-1] + t_edges[1:])
    g_sum   = np.zeros(t_centres.size, dtype=np.float64)
    g_count = np.zeros(t_centres.size, dtype=np.int64)

    # Production grid is fixed; rounding to 5 dp (~1 m) is just defensive.
    keys = (np.round(lat, 5) * 1_000_000).astype(np.int64) * 100_000_000 \
         + (np.round(lon, 5) * 1_000_000).astype(np.int64)
    order = np.argsort(keys, kind='stable')
    starts, ends = _group_boundaries(keys[order])
    for s, e in zip(starts, ends):
        m = e - s
        if m < 2:
            continue
        idx = order[s:e]
        n_pairs = min(max_pairs_per_cell, m * (m - 1) // 2)
        ii = rng.integers(0, m, size=n_pairs)
        jj = rng.integers(0, m, size=n_pairs)
        keep = ii != jj
        a, b = idx[ii[keep]], idx[jj[keep]]
        h_t   = np.abs(t_h[a] - t_h[b])
        gamma = 0.5 * (val[a] - val[b]) ** 2
        bin_idx = np.clip(np.digitize(h_t, t_edges) - 1, 0, t_centres.size - 1)
        np.add.at(g_sum,   bin_idx, gamma)
        np.add.at(g_count, bin_idx, 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        return t_centres, np.where(g_count > 0, g_sum / g_count, np.nan), g_count


def empirical_st_12x12(lat, lon, t_h, val,
                       *, n_pairs=B1_VARIOGRAM_SUBSAMPLE, rng_seed=0):
    """Same kernel as kriging._empirical_st_variogram but on a 12×12 grid."""
    rng = np.random.default_rng(rng_seed)
    n = val.size
    n_target = min(n_pairs, n * (n - 1) // 2)
    i = rng.integers(0, n, size=n_target)
    j = rng.integers(0, n, size=n_target)
    keep = i != j
    i, j = i[keep], j[keep]
    h_s   = haversine_km(lat[i], lon[i], lat[j], lon[j])
    h_t   = np.abs(t_h[i] - t_h[j])
    gamma = 0.5 * (val[i] - val[j]) ** 2
    s_edges = np.linspace(0, B1_W_SPACE_KM * 1.5, 13)
    t_edges = np.linspace(0, B1_W_TIME_H  * 1.5, 13)
    s_c = 0.5 * (s_edges[:-1] + s_edges[1:])
    t_c = 0.5 * (t_edges[:-1] + t_edges[1:])
    s_idx = np.clip(np.digitize(h_s, s_edges) - 1, 0, s_c.size - 1)
    t_idx = np.clip(np.digitize(h_t, t_edges) - 1, 0, t_c.size - 1)
    g_sum   = np.zeros((s_c.size, t_c.size), dtype=np.float64)
    g_count = np.zeros_like(g_sum, dtype=np.int64)
    np.add.at(g_sum,   (s_idx, t_idx), gamma)
    np.add.at(g_count, (s_idx, t_idx), 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        g_mean = np.where(g_count > 0, g_sum / g_count, np.nan)
    return s_c, t_c, g_mean, g_count, s_edges, t_edges


def plot_strict_diagnostics(lat, lon, t_h, val, label):
    s_c12, t_c12, g12, n12, s_edges, t_edges = empirical_st_12x12(lat, lon, t_h, val)
    s_c, g_space, n_space = same_slot_variogram(lat, lon, t_h, val, s_edges=s_edges)
    t_c, g_time,  n_time  = same_cell_variogram(lat, lon, t_h, val, t_edges=t_edges)

    print(f'[{label}] same-slot pairs: {n_space.sum():>10,d}  '
          f'({(n_space > 0).sum()}/{s_c.size} bins populated)')
    print(f'[{label}] same-cell pairs: {n_time.sum():>10,d}  '
          f'({(n_time  > 0).sum()}/{t_c.size} bins populated)')
    print(f'[{label}] 12×12 joint   : {n12.sum():>10,d}  '
          f'({(n12     > 0).sum()}/{g12.size} bins populated)')

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))
    axes[0].plot(s_c, g_space, marker='o')
    axes[0].set_xlabel('h_s (km)')
    axes[0].set_ylabel('γ')
    axes[0].set_title(f'{label}: γ(h_s) | h_t = 0  (same-slot)')
    axes[0].grid(alpha=0.3)

    axes[1].plot(t_c, g_time, marker='o', color='tab:orange')
    axes[1].set_xlabel('h_t (hours)')
    axes[1].set_ylabel('γ')
    axes[1].set_title(f'{label}: γ(h_t) | h_s = 0  (same-cell)')
    axes[1].grid(alpha=0.3)

    vmax = np.nanquantile(g12, 0.99)
    im = axes[2].pcolormesh(t_c12, s_c12, g12, shading='auto',
                             cmap='viridis', vmax=vmax)
    axes[2].set_xlabel('h_t (hours)')
    axes[2].set_ylabel('h_s (km)')
    axes[2].set_title(f'{label}: γ(h_s, h_t)  12×12  (clip @ p99 {vmax:.3f})')
    plt.colorbar(im, ax=axes[2], label='γ')
    plt.tight_layout(); plt.show()

    return {
        's_centres': s_c, 'g_space': g_space, 'n_space': n_space,
        't_centres': t_c, 'g_time':  g_time,  'n_time':  n_time,
        'g_12x12':   g12, 'n_12x12': n12,
        's_centres_12': s_c12, 't_centres_12': t_c12,
    }


strict_raw = plot_strict_diagnostics(lat_raw, lon_raw, t_h_raw, val,       label='raw AOD')
strict_res = plot_strict_diagnostics(lat,     lon,     t_h,     val_resid, label='residual (AOD − CAMS)')

## 5. Fit the new metric variogram on residuals

This is the production fit — exactly what `kriging.fit_and_save_variogram`
runs.  After Sections 2–4 confirmed the structural fixes, the metric variogram
is a 4-parameter model (`metric.var`, `metric.len_scale`, `metric.nugget`,
`k_km_per_hour`) and the residual surface should be well-conditioned enough to
fit cleanly.

In [ ]:
# Fit the production metric variogram and pull the empirical grid back for
# side-by-side plotting in §6.

def fit_with_diagnostic(obs_lat, obs_lon, obs_time_h, obs_val):
    s_c, t_c, g_grid, g_cnt = kriging._empirical_st_variogram(
        obs_lat, obs_lon, obs_time_h, obs_val,
        n_mixed_pairs=B1_VARIOGRAM_SUBSAMPLE,
        return_counts=True,
    )
    vgm = fit_variogram(obs_lat, obs_lon, obs_time_h, obs_val)
    return vgm, {
        's_centres': s_c, 't_centres': t_c,
        'g_grid': g_grid, 'g_count': g_cnt,
    }


def summarise(name, vgm, diag):
    print(f'\n[{name}]')
    print(f'  metric: var={vgm.metric.var:.4f}   '
          f'len_scale={vgm.metric.len_scale:.1f} km   '
          f'nugget={vgm.metric.nugget:.4f}')
    print(f'  k       {vgm.k_km_per_hour:.4f} km/h')
    print(f'  total sill={vgm.total_sill():.4f}')
    cnt = diag['g_count']
    g   = diag['g_grid']
    valid = np.isfinite(g)
    print(f'  empirical: {valid.sum()}/{g.size} bins ≥ {B1_VARIOGRAM_MIN_BIN_PAIRS} pairs '
          f'(total pairs in fit: {int(cnt[valid].sum()):,d})')

In [ ]:
vgm_res, diag_res = fit_with_diagnostic(lat, lon, t_h, val_resid)
summarise('residual (production target)', vgm_res, diag_res)

In [ ]:
# For comparison, fit the raw-AOD pool the same way.
vgm_raw, diag_raw = fit_with_diagnostic(lat_raw, lon_raw, t_h_raw, val)
summarise('raw AOD (comparison)', vgm_raw, diag_raw)

## 6. Empirical vs fitted — side by side

The fitted metric surface should hug the empirical one across the bins that
entered the fit (white squares = bins dropped for fewer than
`B1_VARIOGRAM_MIN_BIN_PAIRS` pairs).  Big residuals near the empty cells are
expected and harmless — those bins were intentionally excluded.

In [ ]:
def plot_fit_vs_empirical(vgm, diag, title):
    s_c, t_c = diag['s_centres'], diag['t_centres']
    g_emp = diag['g_grid']
    S, T  = np.meshgrid(s_c, t_c, indexing='ij')
    g_pred = vgm.gamma(S, T)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
    vmax = np.nanquantile(g_emp, 0.99)
    plot_empirical(g_emp,  s_c, t_c, f'{title}: empirical',          vmax=vmax, ax=axes[0])
    plot_empirical(g_pred, s_c, t_c, f'{title}: fitted',             vmax=vmax, ax=axes[1])
    plot_empirical(g_pred - g_emp, s_c, t_c, f'{title}: fitted − emp', ax=axes[2])
    plt.tight_layout(); plt.show()

plot_fit_vs_empirical(vgm_res, diag_res, 'residual')
plot_fit_vs_empirical(vgm_raw, diag_raw, 'raw AOD')

## 7. Save the chosen variogram (manual step)

**Do not run this cell blindly.** Inspect Sections 5–6 first.

* `vgm_res` is the production fit (residual target).  Saving it under the
  production path overwrites `MODELS_DIR/st_variogram.json`, which is what the
  workers load.
* `vgm_raw` is kept for back-comparison only — saving it would silently switch
  the production target back to raw AOD without updating
  `config.B1_VARIOGRAM_TARGET`, which would mismatch the loader.

In [ ]:
# Uncomment ONE of the two lines below after inspecting the plots.

# save_variogram(vgm_res)                                                   # production path
# save_variogram(vgm_raw, path=MODELS_DIR / 'st_variogram_raw.json')        # side-channel for raw-AOD fit

print('Nothing saved. Edit this cell to commit a choice.')